# Notebook 09 — Laplace Transforms and Transfer Functions

**Companion to Chapter 9**

This notebook translates the local vertical-motion model from state space into the Laplace domain. It verifies signs, poles, initial-condition effects, and the equivalence of state-space and transfer-function simulations.

## Learning objectives

- construct a transfer function from a differential equation and a state-space model;
- interpret poles as the same modes found by eigenvalue analysis;
- distinguish zero-state input response from zero-input initial-condition response;
- verify equivalent numerical representations of the same plant.

## 1. The local vertical plant

Depth $z$ is positive downward, velocity $v$ is positive upward, and control force $u$ is positive upward:

$$\delta\dot z=-\delta v,\qquad \delta\dot v=a\,\delta z+\frac{u}{m},\qquad a<0.$$

The sign convention matters: a positive upward force must reduce depth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})

m = 85.0                 # kg
rho = 1025.0             # kg/m^3
g = 9.80665              # m/s^2
z_star = 20.0            # m, positive downward
V_g0 = 8.0e-3            # m^3 at the surface
p_atm = 101325.0          # Pa
p_star = p_atm + rho*g*z_star
V_g_star = V_g0*p_atm/p_star
k_B = -rho**2*g**2*V_g_star/p_star
a = k_B/m

A = np.array([[0.0, -1.0], [a, 0.0]])
B = np.array([[0.0], [1.0/m]])         # positive input force is upward
C = np.array([[1.0, 0.0]])
D = np.zeros((1, 1))

print(f"Local buoyancy slope k_B = {k_B:.4f} N/m")
print(f"Plant coefficient a = {a:.6f} s^-2")

## 2. From state space to a transfer function

With zero initial conditions,

$$G_{zu}(s)=\frac{Z(s)}{U(s)}=-\frac{1}{m(s^2+a)}.$$

The negative numerator is a physical sign, not a software convention.

In [ ]:
num, den = signal.ss2tf(A, B, C, D)
num = num[0]
poles = np.roots(den)
print("Numerator:", num)
print("Denominator:", den)
print("Poles:", poles)
assert np.allclose(num, [0, 0, -1/m], atol=1e-12)
assert np.allclose(np.sort(poles), np.sort(np.linalg.eigvals(A)))

## 3. Poles and the stability result from Chapter 6

The characteristic equation is $s^2+a=0$. Because $a<0$, the poles are real and of opposite sign. The positive pole is the same unstable mode previously obtained as an eigenvalue of $A$. A transform has changed the representation, not the physics.

In [ ]:
growth_time = 1/max(poles.real)
print(f"Unstable-mode e-folding time: {growth_time:.2f} s")

fig, ax = plt.subplots()
ax.scatter(poles.real, poles.imag, s=80)
ax.axvline(0, color="k", lw=1)
ax.set(xlabel="Real part [s$^{-1}$]", ylabel="Imaginary part [s$^{-1}$]", title="Open-loop poles")
plt.show()

## 4. Equivalent zero-state simulations

A short force pulse is applied to both representations. Agreement is a useful implementation test; it also catches the most common error in this model—a missing minus sign between upward velocity and increasing depth.

In [ ]:
t = np.linspace(0, 20, 1201)
u = np.where((t >= 1.0) & (t < 2.0), 10.0, 0.0)
sys_ss = signal.StateSpace(A, B, C, D)
sys_tf = signal.TransferFunction([-1/m], [1, 0, a])
_, y_ss, _ = signal.lsim(sys_ss, U=u, T=t)
_, y_tf, _ = signal.lsim(sys_tf, U=u, T=t)
assert np.max(np.abs(y_ss-y_tf)) < 1e-9

fig, ax = plt.subplots()
ax.plot(t, y_ss, label="state space")
ax.plot(t, y_tf, "--", label="transfer function")
ax.set(xlabel="Time [s]", ylabel="Depth perturbation [m]", title="Equivalent input–output response")
ax.legend(); plt.show()

## 5. Initial conditions are a separate response

A transfer function is defined with zero initial conditions. It does not erase initial conditions from the physical system; their contribution must be added as a zero-input response.

In [ ]:
x0 = np.array([0.25, 0.0])
_, y_ic, x_ic = signal.lsim(sys_ss, U=np.zeros_like(t), T=t, X0=x0)
fig, ax = plt.subplots()
ax.plot(t, y_ic)
ax.set(xlabel="Time [s]", ylabel="Depth perturbation [m]", title="Zero-input response from a depth offset")
plt.show()

## 6. A stable reference system

Step-response language is easiest to interpret for a stable system. The first-order model $G(s)=1/(5s+1)$ settles to unit gain, while the diver plant does not settle open loop.

In [ ]:
stable = signal.TransferFunction([1], [5, 1])
t_ref, y_step = signal.step(stable, T=np.linspace(0, 30, 500))
fig, ax = plt.subplots()
ax.plot(t_ref, y_step)
ax.axhline(1, color="k", ls=":")
ax.set(xlabel="Time [s]", ylabel="Output", title="Stable first-order step response")
plt.show()

## Engineering exercises

1. Double the mass while holding $a$ fixed. Which coefficients and responses change?
2. Reverse the input sign in $B$. Explain the resulting numerator physically.
3. Use $x_0=[0,0.1]^T$ and predict the initial slope of depth before simulating.
4. Derive $G_{vu}(s)$ and verify it using a different output matrix.

## Summary

The transfer function and state-space model encode the same local dynamics. Poles reproduce the plant eigenvalues, while the numerator encodes the direction from input force to measured depth. Chapter 10 uses this input–output view to close the feedback loop.